# 02 — Exploratory Data Analysis
**Meridian Governance Group — AI Policy Research Agent**

This notebook explores the document corpus and the chunk/embedding space that powers the agent's retrieval. It assumes `01_data_pipeline.ipynb` has been run.

> **AI usage disclosure:** Scaffolding developed with Anthropic Claude (Claude Code); reviewed by the author.

In [ ]:
%pip install sentence-transformers scikit-learn matplotlib seaborn pandas numpy

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.vector_store import SimpleVectorStore

sns.set_theme(style="whitegrid")

# Load the numpy store if Step 6 of notebook 01 was run.
# If not (pure Databricks run), rebuild it in memory from the Delta table.
if config.VECTOR_STORE_PATH.exists():
    store = SimpleVectorStore.load(config.VECTOR_STORE_PATH, config.VECTOR_STORE_META_PATH)
    print(f"Loaded {len(store):,} chunks from local store.")
else:
    print("Local store not found — rebuilding from Delta table...")
    from sentence_transformers import SentenceTransformer

    _SOURCE_TO_DOC_ID = {
        "nist_ai_rmf":   "nist_ai_rmf_1_0",
        "nist_ai_600_1": "nist_ai_600_1",
        "eu_ai_act":     "eu_ai_act",
    }
    rows = spark.sql("SELECT source, text FROM main.default.ai_governance_chunks").toPandas()
    texts     = rows["text"].tolist()
    metadatas = [{"doc_id": _SOURCE_TO_DOC_ID.get(s, s)} for s in rows["source"]]

    embedder   = SentenceTransformer(config.EMBEDDING_MODEL)
    embeddings = embedder.encode(texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)

    store = SimpleVectorStore(config.EMBEDDING_DIM)
    store.add(embeddings, texts, metadatas)
    print(f"Built {len(store):,} chunks from Delta table.")


## 1. Corpus overview
Chunks and characters per document.

In [ ]:
df = pd.DataFrame(store.metadatas)
df["chars"] = [len(t) for t in store.texts]
df["words"] = [len(t.split()) for t in store.texts]

# Derive human-readable name from doc_id (page info is not stored in metadata).
df["short_name"] = df["doc_id"].map(config.DOC_SHORT_NAMES)

summary = df.groupby("short_name").agg(
    chunks=("chars", "size"),
    total_chars=("chars", "sum"),
    avg_chars=("chars", "mean"),
).round(1)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
summary["chunks"].plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Chunks per document"); axes[0].set_ylabel("chunks")
axes[0].tick_params(axis="x", rotation=20)
summary["total_chars"].plot(kind="bar", ax=axes[1], color="#55A868")
axes[1].set_title("Total characters per document"); axes[1].set_ylabel("chars")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

## 2. Chunk-length distribution
Confirms the splitter behaved (~1000 chars).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(df, x="chars", hue="short_name", bins=40, ax=ax)
ax.axvline(config.CHUNK_SIZE, color="red", ls="--", label="target size")
ax.set_title("Chunk length distribution"); ax.set_xlabel("characters")
ax.legend(); plt.tight_layout(); plt.show()
df["chars"].describe().round(1)

## 3. Term frequency
Most frequent meaningful terms across the corpus (simple stopword-filtered word counts).

In [ ]:
import re
from collections import Counter

STOP = set("""the a an and or of to in for on with by is are be as at this that \
shall may will can which such from any all not no its it their these those been \
has have had also other than into under per via where when who whom whose if then \
use used using based within without between among about above below more most each""".split())

def tokenize(text: str):
    return [w for w in re.findall(r"[a-zA-Z]{3,}", text.lower()) if w not in STOP]

counts = Counter()
for t in store.texts:
    counts.update(tokenize(t))
top = pd.DataFrame(counts.most_common(20), columns=["term", "count"])

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(top, y="term", x="count", ax=ax, color="#C44E52")
ax.set_title("Top 20 terms across the policy corpus")
plt.tight_layout(); plt.show()

In [ ]:
# Word cloud (optional, visual). Skips gracefully if wordcloud isn't installed.
try:
    from wordcloud import WordCloud
    wc = WordCloud(width=900, height=400, background_color="white")
    wc.generate_from_frequencies(counts)
    plt.figure(figsize=(11, 5)); plt.imshow(wc); plt.axis("off")
    plt.title("Corpus word cloud"); plt.show()
except Exception as e:
    print("wordcloud unavailable:", e)

## 4. Embedding space (t-SNE)
Project the 384-dim chunk embeddings to 2-D to see whether documents form distinct clusters — a good sign that retrieval can distinguish them.

In [ ]:
from sklearn.manifold import TSNE

emb = store.embeddings
perplexity = min(30, max(5, len(store) // 10))
coords = TSNE(n_components=2, perplexity=perplexity, init="pca",
              random_state=42).fit_transform(emb)

plot_df = pd.DataFrame(coords, columns=["x", "y"])
plot_df["document"] = df["short_name"].values
fig, ax = plt.subplots(figsize=(9, 7))
sns.scatterplot(plot_df, x="x", y="y", hue="document", s=18, alpha=0.7, ax=ax)
ax.set_title("t-SNE of chunk embeddings by document")
plt.tight_layout(); plt.show()

## 5. Inter-document similarity
Mean cosine similarity between each pair of documents' chunks — quantifies topical overlap.

In [ ]:
docs = [d['doc_id'] for d in config.DOCUMENTS]
names = [config.DOC_SHORT_NAMES[d] for d in docs]
idx_by_doc = {d: np.array([i for i, m in enumerate(store.metadatas) if m['doc_id'] == d]) for d in docs}

mat = np.zeros((len(docs), len(docs)))
for i, da in enumerate(docs):
    for j, db in enumerate(docs):
        A, B = emb[idx_by_doc[da]], emb[idx_by_doc[db]]
        mat[i, j] = float((A @ B.T).mean())  # vectors are normalized

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(mat, annot=True, fmt=".3f", xticklabels=names, yticklabels=names,
            cmap="viridis", ax=ax)
ax.set_title("Mean inter-document cosine similarity")
plt.tight_layout(); plt.show()

### EDA takeaways
- The three documents differ in size; chunk counts reflect that.
- Chunk lengths cluster near the 1000-char target as designed.
- t-SNE shows document-level structure, so semantic retrieval can distinguish sources.
- The two NIST documents are more similar to each other than to the EU AI Act, matching their shared origin and vocabulary.

Proceed to `03_agent_evaluation.ipynb`.